In [1]:
import pandas as pd

In [2]:
import pandas as pd

season = "2627"
praefix = season[:2]  
leagues = {
    "pl": "E0",  
    "de": "D1",  
    "es": "SP1",  
    "it": "I1",  
    "fr": "F1", 
}
urls = {
    name: f"https://football-data.co.uk/mmz4281/{season}/{code}.csv"
    for name, code in leagues.items()
}

In [3]:
import numpy as np


In [4]:
def snake_case(df):
    df = df.copy()
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )
    return df


In [5]:
column_mapping = {
    "hometeam": "home_team",
    "awayteam": "away_team",
    "fthg": "full_time_home_goals",
    "ftag": "full_time_away_goals",
    "ftr": "full_time_result",
    "hthg": "half_time_home_goals",
    "htag": "half_time_away_goals",
    "htr": "half_time_result",
    "hs": "home_shots",
    "as": "away_shots",
    "hst": "home_shots_on_target",
    "ast": "away_shots_on_target"
}



In [6]:
columns_for_standings = [
    "date",
    "home_team",
    "away_team",
    "full_time_home_goals",
    "full_time_away_goals",
    "full_time_result",
]

In [7]:
league_data = {}

for name, code in leagues.items():
    league_data[name] = {}

    full = pd.read_csv(urls[name], encoding="utf-8-sig")
    full = snake_case(full)
    full = full.rename(columns=column_mapping)

    league_data[name]["full"] = full
    league_data[name]["matches"] = full[columns_for_standings].copy()

    matches = league_data[name]["matches"]

    # Combine home and away teams, remove duplicates, then sort them.
    league_data[name]["teams"] = sorted(
        set(matches["home_team"]) | set(matches["away_team"])
    )

    league_data[name]["team_matches"] = {}

    for team in league_data[name]["teams"]:
        team_matches = matches[
            (matches["home_team"] == team)
            | (matches["away_team"] == team)
        ].copy()

        league_data[name]["team_matches"][team] = team_matches


In [8]:
for name in leagues:
    standings_rows = []

    for team in league_data[name]["teams"]:
        club_df = league_data[name]["team_matches"][team]

        
        played = club_df.dropna(
            subset=["full_time_home_goals", "full_time_away_goals"]
        )

        home_games = played[played["home_team"] == team]
        away_games = played[played["away_team"] == team]

        home_wins = (
            home_games["full_time_home_goals"]
            > home_games["full_time_away_goals"]
        ).sum()
        away_wins = (
            away_games["full_time_away_goals"]
            > away_games["full_time_home_goals"]
        ).sum()
        wins = home_wins + away_wins

        home_draws = (
            home_games["full_time_home_goals"]
            == home_games["full_time_away_goals"]
        ).sum()
        away_draws = (
            away_games["full_time_away_goals"]
            == away_games["full_time_home_goals"]
        ).sum()
        draws = home_draws + away_draws

        home_losses = (
            home_games["full_time_home_goals"]
            < home_games["full_time_away_goals"]
        ).sum()
        away_losses = (
            away_games["full_time_away_goals"]
            < away_games["full_time_home_goals"]
        ).sum()
        losses = home_losses + away_losses

        goals_for = (
            home_games["full_time_home_goals"].sum()
            + away_games["full_time_away_goals"].sum()
        )
        goals_against = (
            home_games["full_time_away_goals"].sum()
            + away_games["full_time_home_goals"].sum()
        )
        goal_difference = goals_for - goals_against
        points = 3 * wins + draws

        standings_rows.append({
            "Team": team,
            "Played": len(played),
            "W": wins,
            "D": draws,
            "L": losses,
            "GF": goals_for,
            "GA": goals_against,
            "GD": goal_difference,
            "Points": points,
        })

    league_data[name]["standings"] = (
        pd.DataFrame(standings_rows)
        .sort_values(
            ["Points", "GD", "GF"],
            ascending=False,
        )
        .reset_index(drop=True)
    )


In [ ]:
for name in leagues:
    standings_rows = []

    for team in league_data[name]["teams"]:
        club_df = league_data[name]["team_matches"][team]

        
        played = club_df.dropna(
            subset=["full_time_home_goals", "full_time_away_goals"]
        )

        home_games = played[played["home_team"] == team]
        away_games = played[played["away_team"] == team]

        home_wins = (
            home_games["full_time_home_goals"]
            > home_games["full_time_away_goals"]
        ).sum()
        away_wins = (
            away_games["full_time_away_goals"]
            > away_games["full_time_home_goals"]
        ).sum()
        wins = home_wins + away_wins

        home_draws = (
            home_games["full_time_home_goals"]
            == home_games["full_time_away_goals"]
        ).sum()
        away_draws = (
            away_games["full_time_away_goals"]
            == away_games["full_time_home_goals"]
        ).sum()
        draws = home_draws + away_draws

        home_losses = (
            home_games["full_time_home_goals"]
            < home_games["full_time_away_goals"]
        ).sum()
        away_losses = (
            away_games["full_time_away_goals"]
            < away_games["full_time_home_goals"]
        ).sum()
        losses = home_losses + away_losses

        goals_for = (
            home_games["full_time_home_goals"].sum()
            + away_games["full_time_away_goals"].sum()
        )
        goals_against = (
            home_games["full_time_away_goals"].sum()
            + away_games["full_time_home_goals"].sum()
        )
        goal_difference = goals_for - goals_against
        points = 3 * wins + draws

        standings_rows.append({
            "Team": team,
            "Played": len(played),
            "W": wins,
            "D": draws,
            "L": losses,
            "GF": goals_for,
            "GA": goals_against,
            "GD": goal_difference,
            "Points": points,
        })

    league_data[name]["standings"] = (
        pd.DataFrame(standings_rows)
        .sort_values(
            ["Points", "GD", "GF"],
            ascending=False,
        )
        .reset_index(drop=True)
    )


,Team,Played,W,D,L,GF,GA,GD,Points
1,Arsenal,10,9,0,1,20,6,14,27
2,Brighton,10,7,1,2,30,12,18,22
3,Man City,10,7,0,3,22,12,10,21
4,Liverpool,10,5,2,2,8,8,0,17
5,Chelsea,10,5,1,4,26,16,10,16
6,Bournemouth,10,4,2,3,14,20,-6,14
7,Everton,10,4,1,2,16,10,6,13
8,Leeds,10,3,3,2,14,10,4,12
9,Newcastle,10,3,2,4,12,18,-6,11
10,Aston Villa,10,3,2,3,12,20,-8,11


In [17]:
standings_dfs = {}
for name in leagues:
    standings_dfs[name] = league_data[name]["standings"]


In [20]:
full_dfs = {}

for name in leagues:
    full_dfs[name] = league_data[name]["full"]

In [30]:
full_dfs["pl"]
step = 10
for start in range(80, len(full_dfs["pl"].columns), step):
    blok = full_dfs["pl"].columns[start : start + step]
    print(f"\n--- Kolone {start} do {start + len(blok) - 1} ---")
    print(full_dfs["pl"][blok].head(3))


--- Kolone 80 do 89 ---
   bwcd   bwca  ppch  ppcd   ppca  skbch  skbcd  skbca  maxch  maxcd
0  6.50  11.50  1.17   7.0  15.00   1.17    7.5  14.00   1.22    7.5
1  4.80   1.36  8.50   5.0   1.33   8.50    5.0   1.33   9.00    5.5
2  3.25   3.00  2.25   3.3   3.10   2.25    3.4   3.10   2.38    3.5

--- Kolone 90 do 99 ---
   maxca  avgch  avgcd  avgca  bfech  bfecd  bfeca  b365c>2.5  b365c<2.5  maxc>2.5
0  16.00   1.20   6.72  14.00   1.23   7.40  18.00       1.57       2.38      1.58
1   1.37   8.49   5.11   1.34   9.60   5.40   1.39       1.57       2.38      1.58
2   3.20   2.32   3.36   3.05   2.42   3.45   3.25       1.98       1.88      1.98

--- Kolone 100 do 109 ---
   maxc<2.5  avgc>2.5  avgc<2.5  bfec>2.5  bfec<2.5  ahch  b365cahh  b365caha  maxcahh  maxcaha
0      2.45      1.55      2.37      1.63      2.56 -2.00      2.05      1.75     2.05     1.83
1      2.50      1.55      2.38      1.63      2.56  1.50      1.88      1.98     1.88     1.98
2      1.88      1.93      

In [32]:
column_mapping = {
    # Statističke i meč kolone (iz tvog prvog bloka)
    "hxg": "home_xg",  # Expected Goals (xG) domaćina
    "axg": "away_xg",  # Expected Goals (xG) gosta
    "home_shots": "home_shots",  # Ukupni šutevi domaćina (u sirovom fajlu često 'HS')
    "away_shots": "away_shots",  # Ukupni šutevi gosta (u sirovom fajlu često 'AS')
    "home_shots_on_target": "home_shots_on_target",  # Šutevi u okvir gola domaćina (često 'HST')
    "away_shots_on_target": "away_shots_on_target",  # Šutevi u okvir gola gosta (često 'AST')
    "hf": "home_fouls",  # Faulovi domaćina
    "af": "away_fouls",  # Faulovi gosta
    # Kolone 20 do 29
    "hc": "home_corners",  # Korneri domaćina
    "ac": "away_corners",  # Korneri gosta
    "hy": "home_yellow_cards",  # Žuti kartoni domaćina
    "ay": "away_yellow_cards",  # Žuti kartoni gosta
    "hr": "home_red_cards",  # Crveni kartoni domaćina
    "ar": "away_red_cards",  # Crveni kartoni gosta
    "b365h": "b365_home_odds",  # Bet365 kvota – pobjeda domaćina (Home)
    "b365d": "b365_draw_odds",  # Bet365 kvota – neriješeno (Draw)
    "b365a": "b365_away_odds",  # Bet365 kvota – pobjeda gosta (Away)
    "bfdh": "bfd_home_odds",  # Betfair kvota – pobjeda domaćina
    # Kolone 30 do 39
    "bfdd": "bfd_draw_odds",  # Betfair kvota – neriješeno
    "bfda": "bfd_away_odds",  # Betfair kvota – pobjeda gosta
    "bvh": "bv_home_odds",  # BetVictor kvota – pobjeda domaćina
    "bvd": "bv_draw_odds",  # BetVictor kvota – neriješeno
    "bva": "bv_away_odds",  # BetVictor kvota – pobjeda gosta
    "bwh": "bw_home_odds",  # Bwin kvota – pobjeda domaćina
    "bwd": "bw_draw_odds",  # Bwin kvota – neriješeno
    "bwa": "bw_away_odds",  # Bwin kvota – pobjeda gosta
    "pph": "pp_home_odds",  # Paddy Power kvota – pobjeda domaćina
    "ppd": "pp_draw_odds",  # Paddy Power kvota – neriješeno
    # Kolone 40 do 49
    "ppa": "pp_away_odds",  # Paddy Power – pobjeda gosta
    "skbh": "skb_home_odds",  # Sky Bet – pobjeda domaćina
    "skbd": "skb_draw_odds",  # Sky Bet – neriješeno
    "skba": "skb_away_odds",  # Sky Bet – pobjeda gosta
    "maxh": "max_home_odds",  # Najveća tržišna kvota – domaćin
    "maxd": "max_draw_odds",  # Najveća tržišna kvota – neriješeno
    "maxa": "max_away_odds",  # Najveća tržišna kvota – gost
    "avgh": "avg_home_odds",  # Prosječna tržišna kvota – domaćin
    "avgd": "avg_draw_odds",  # Prosječna tržišna kvota – neriješeno
    "avga": "avg_away_odds",  # Prosječna tržišna kvota – gost
    # Kolone 50 do 59
    "bfeh": "bfe_home_odds",  # Betfair Exchange – domaćin
    "bfed": "bfe_draw_odds",  # Betfair Exchange – neriješeno
    "bfea": "bfe_away_odds",  # Betfair Exchange – gost
    "b365>2.5": "b365_over_2_5_odds",  # Bet365 – više od 2.5 gola
    "b365<2.5": "b365_under_2_5_odds",  # Bet365 – manje od 2.5 gola
    "max>2.5": "max_over_2_5_odds",  # Najveća kvota – više od 2.5 gola
    "max<2.5": "max_under_2_5_odds",  # Najveća kvota – manje od 2.5 gola
    "avg>2.5": "avg_over_2_5_odds",  # Prosječna kvota – više od 2.5 gola
    "avg<2.5": "avg_under_2_5_odds",  # Prosječna kvota – manje od 2.5 gola
    "bfe>2.5": "bfe_over_2_5_odds",  # Betfair Exchange – više od 2.5 gola
    # Kolone 60 do 69
    "bfe<2.5": "bfe_under_2_5_odds",  # Betfair Exchange – manje od 2.5 gola
    "ahh": "asian_handicap_line",  # Linija azijskog hendikepa (Home)
    "b365ahh": "b365_ah_home_odds",  # Bet365 Asian Handicap – domaćin
    "b365aha": "b365_ah_away_odds",  # Bet365 Asian Handicap – gost
    "maxahh": "max_ah_home_odds",  # Najveća kvota Asian Handicap – domaćin
    "maxaha": "max_ah_away_odds",  # Najveća kvota Asian Handicap – gost
    "avgahh": "avg_ah_home_odds",  # Prosječna kvota Asian Handicap – domaćin
    "avgaha": "avg_ah_away_odds",  # Prosječna kvota Asian Handicap – gost
    "bfeahh": "bfe_ah_home_odds",  # Betfair Exchange Asian Handicap – domaćin
    "bfeaha": "bfe_ah_away_odds",  # Betfair Exchange Asian Handicap – gost
    # Kolone 70 do 79
    "b365ch": "b365_close_home_odds",  # Bet365 closing kvota – domaćin
    "b365cd": "b365_close_draw_odds",  # Bet365 closing kvota – neriješeno
    "b365ca": "b365_close_away_odds",  # Bet365 closing kvota – gost
    "bfdch": "bfd_close_home_odds",  # Betfair closing kvota – domaćin
    "bfdcd": "bfd_close_draw_odds",  # Betfair closing kvota – neriješeno
    "bfdca": "bfd_close_away_odds",  # Betfair closing kvota – gost
    "bvch": "bv_close_home_odds",  # BetVictor closing kvota – domaćin
    "bvcd": "bv_close_draw_odds",  # BetVictor closing kvota – neriješeno
    "bvca": "bv_close_away_odds",  # BetVictor closing kvota – gost
    "bwch": "bw_close_home_odds",  # Bwin closing kvota – domaćin
    # Kolone 80 do 89
    "bwcd": "bw_close_draw_odds",  # Bwin closing – neriješeno
    "bwca": "bw_close_away_odds",  # Bwin closing – pobjeda gosta
    "ppch": "pp_close_home_odds",  # Paddy Power closing – pobjeda domaćina
    "ppcd": "pp_close_draw_odds",  # Paddy Power closing – neriješeno
    "ppca": "pp_close_away_odds",  # Paddy Power closing – pobjeda gosta
    "skbch": "skb_close_home_odds",  # Sky Bet closing – pobjeda domaćina
    "skbcd": "skb_close_draw_odds",  # Sky Bet closing – neriješeno
    "skbca": "skb_close_away_odds",  # Sky Bet closing – pobjeda gosta
    "maxch": "max_close_home_odds",  # Najveća closing kvota – domaćin
    "maxcd": "max_close_draw_odds",  # Najveća closing kvota – neriješeno
    # Kolone 90 do 99
    "maxca": "max_close_away_odds",  # Najveća closing kvota – gost
    "avgch": "avg_close_home_odds",  # Prosječna closing kvota – domaćin
    "avgcd": "avg_close_draw_odds",  # Prosječna closing kvota – neriješeno
    "avgca": "avg_close_away_odds",  # Prosječna closing kvota – gost
    "bfech": "bfe_close_home_odds",  # Betfair Exchange closing – domaćin
    "bfecd": "bfe_close_draw_odds",  # Betfair Exchange closing – neriješeno
    "bfeca": "bfe_close_away_odds",  # Betfair Exchange closing – gost
    "b365c>2.5": "b365_close_over_2_5_odds",  # Bet365 closing – više od 2.5 gola
    "b365c<2.5": "b365_close_under_2_5_odds",  # Bet365 closing – manje od 2.5 gola
    "maxc>2.5": "max_close_over_2_5_odds",  # Najveća closing kvota – više od 2.5 gola
    # Kolone 100 do 109
    "maxc<2.5": "max_close_under_2_5_odds",  # Najveća closing kvota – manje od 2.5 gola
    "avgc>2.5": "avg_close_over_2_5_odds",  # Prosječna closing kvota – više od 2.5 gola
    "avgc<2.5": "avg_close_under_2_5_odds",  # Prosječna closing kvota – manje od 2.5 gola
    "bfec>2.5": "bfe_close_over_2_5_odds",  # Betfair Exchange closing – više od 2.5 gola
    "bfec<2.5": "bfe_close_under_2_5_odds",  # Betfair Exchange closing – manje od 2.5 gola
    "ahch": "asian_handicap_close_line",  # Zaključna linija azijskog hendikepa (Home)
    "b365cahh": "b365_close_ah_home_odds",  # Bet365 closing Asian Handicap – domaćin
    "b365caha": "b365_close_ah_away_odds",  # Betfair/Bet365 closing Asian Handicap – gost
    "maxcahh": "max_close_ah_home_odds",  # Najveća closing kvota AH – domaćin
    "maxcaha": "max_close_ah_away_odds",  # Najveća closing kvota AH – gost
    # Kolone 110 do 113
    "avgcahh": "avg_close_ah_home_odds",  # Prosječna closing kvota AH – domaćin
    "avgcaha": "avg_close_ah_away_odds",  # Prosječna closing kvota AH – gost
    "bfecahh": "bfe_close_ah_home_odds",  # Betfair Exchange closing AH – domaćin
    "bfecaha": "bfe_close_ah_away_odds",  # Betfair Exchange closing AH – gost
}


In [33]:
for name in leagues.keys():
    full_dfs[name] = full_dfs[name].rename(columns=column_mapping)


In [34]:
full_dfs["de"]

,div,date,time,home_team,away_team,full_time_home_goals,full_time_away_goals,full_time_result,half_time_home_goals,half_time_away_goals,half_time_result,home_xg,away_xg,home_shots,away_shots,home_shots_on_target,away_shots_on_target,home_fouls,away_fouls,home_corners,away_corners,home_yellow_cards,away_yellow_cards,home_red_cards,away_red_cards,b365_home_odds,b365_draw_odds,b365_away_odds,bfd_home_odds,bfd_draw_odds,bfd_away_odds,bv_home_odds,bv_draw_odds,bv_away_odds,bw_home_odds,bw_draw_odds,bw_away_odds,pp_home_odds,pp_draw_odds,pp_away_odds,skb_home_odds,skb_draw_odds,skb_away_odds,max_home_odds,max_draw_odds,max_away_odds,avg_home_odds,avg_draw_odds,avg_away_odds,bfe_home_odds,bfe_draw_odds,bfe_away_odds,b365_over_2_5_odds,b365_under_2_5_odds,max_over_2_5_odds,max_under_2_5_odds,avg_over_2_5_odds,avg_under_2_5_odds,bfe_over_2_5_odds,bfe_under_2_5_odds,asian_handicap_line,b365_ah_home_odds,b365_ah_away_odds,max_ah_home_odds,max_ah_away_odds,avg_ah_home_odds,avg_ah_away_odds,bfe_ah_home_odds,bfe_ah_away_odds,b365_close_home_odds,b365_close_draw_odds,b365_close_away_odds,bfd_close_home_odds,bfd_close_draw_odds,bfd_close_away_odds,bv_close_home_odds,bv_close_draw_odds,bv_close_away_odds,bw_close_home_odds,bw_close_draw_odds,bw_close_away_odds,pp_close_home_odds,pp_close_draw_odds,pp_close_away_odds,skb_close_home_odds,skb_close_draw_odds,skb_close_away_odds,max_close_home_odds,max_close_draw_odds,max_close_away_odds,avg_close_home_odds,avg_close_draw_odds,avg_close_away_odds,bfe_close_home_odds,bfe_close_draw_odds,bfe_close_away_odds,b365_close_over_2_5_odds,b365_close_under_2_5_odds,max_close_over_2_5_odds,max_close_under_2_5_odds,avg_close_over_2_5_odds,avg_close_under_2_5_odds,bfe_close_over_2_5_odds,bfe_close_under_2_5_odds,asian_handicap_close_line,b365_close_ah_home_odds,b365_close_ah_away_odds,max_close_ah_home_odds,max_close_ah_away_odds,avg_close_ah_home_odds,avg_close_ah_away_odds,bfe_close_ah_home_odds,bfe_close_ah_away_odds
0,D1,28/08/2026,19:30,Bayern Munich,Stuttgart,5,1,H,1,0,H,4.06,0.78,21,11,9,4,10,17,9,3,1,3,0,0,1.22,7.00,10.00,1.20,7.50,11.00,1.18,7.50,11.00,1.25,7.25,8.75,1.18,7.50,11.00,1.18,8.00,11.00,1.25,8.00,12.00,1.20,7.38,10.37,1.25,8.2,12.00,1.20,4.50,1.20,5.50,1.17,4.78,1.20,5.60,-2.00,1.83,2.03,1.83,2.03,1.79,1.96,1.90,2.07,1.22,7.50,10.00,1.20,7.50,11.00,1.20,7.5,10.50,1.25,7.25,8.75,1.18,7.0,10.00,1.20,7.50,10.00,1.25,7.50,11.00,1.21,7.37,9.88,1.25,8.2,12.50,1.17,5.00,1.18,5.75,1.15,5.03,1.20,5.70,-2.00,1.83,2.03,1.83,2.03,1.77,1.98,1.90,2.10
1,D1,29/08/2026,14:30,Elversberg,Leverkusen,3,2,H,2,0,H,1.27,1.88,11,21,6,4,18,15,4,3,3,2,0,0,5.00,4.50,1.57,5.00,4.50,1.57,4.80,4.20,1.53,5.25,4.50,1.57,5.00,4.40,1.57,5.00,4.33,1.57,5.25,4.60,1.60,4.90,4.41,1.57,5.40,4.8,1.63,1.50,2.63,1.50,2.75,1.45,2.65,1.52,2.82,1.00,1.90,1.95,1.90,1.95,1.83,1.90,1.95,2.00,5.25,4.33,1.57,5.00,4.50,1.57,5.00,4.4,1.57,4.80,4.40,1.60,5.00,4.2,1.57,5.00,4.33,1.57,5.30,4.50,1.60,5.02,4.35,1.58,5.70,4.5,1.64,1.50,2.63,1.50,2.75,1.46,2.64,1.54,2.78,1.00,1.90,1.95,1.90,1.95,1.84,1.90,1.94,2.02
2,D1,29/08/2026,14:30,FC Koln,Hoffenheim,3,2,H,0,1,A,1.95,1.88,18,15,6,4,11,14,6,7,0,2,0,0,2.88,3.75,2.25,2.80,3.75,2.30,2.75,3.60,2.20,2.80,3.70,2.30,2.88,3.80,2.20,2.80,3.75,2.20,2.92,3.80,2.30,2.83,3.69,2.25,3.10,3.9,2.36,1.53,2.50,1.67,2.55,1.53,2.41,1.60,2.60,0.25,1.85,2.00,1.85,2.00,1.80,1.94,1.93,2.05,3.10,3.75,2.15,NaN,NaN,NaN,NaN,NaN,NaN,3.00,3.60,2.20,3.20,3.5,2.10,3.20,3.70,2.10,3.20,3.75,2.20,3.08,3.67,2.14,3.35,3.8,2.24,1.57,2.38,1.60,2.50,1.55,2.40,1.64,2.54,0.25,1.95,1.90,1.95,1.90,1.88,1.86,2.03,1.94
3,D1,29/08/2026,14:30,Mainz,Paderborn,0,0,D,0,0,D,1.23,0.73,21,10,4,0,8,10,11,2,1,2,0,0,1.65,4.00,5.00,1.67,4.20,4.60,1.60,3.80,4.60,1.68,4.00,4.60,1.67,4.00,4.75,1.67,3.90,4.75,1.69,4.20,5.00,1.66,3.98,4.71,1.72,4.3,5.20,1.67,2.20,1.67,2.30,1.62,2.22,1.71,2.36,-0.75,1.83,2.03,1.84,2.03,1.80,1.95,1.90,2.07,1.50,4.50,6.00,1.50,4.60,6.00,1.50,4.6,5.50,1.52,4.60,5.50,1.50,4.2,5.50,1.50,4.50,5.50,1.52,4.60,6.25,1.51,4.49,5.65,1.

# Football project variable guide

## Configuration

| Variable | What it contains |
|---|---|
| `season` | Season string, currently `"2627"` |
| `praefix` | First two characters of `season`; currently `"26"`. Not used in the current data-loading code. |
| `leagues` | Dictionary mapping short league names (`pl`, `de`, `es`, `it`, `fr`) to football-data.co.uk codes |
| `urls` | Dictionary mapping each short league name to its CSV URL |
| `column_mapping` | Dictionary mapping source column names to clearer names |
| `columns_for_standings` | List of match columns used to calculate standings |

## Functions and imports

| Name | What it is |
|---|---|
| `pd` | pandas, imported as `pd` |
| `np` | NumPy, imported as `np`; it does not appear to be used in the current code |
| `snake_case(df)` | Function that cleans DataFrame column names; `df` is its input parameter |

## Main data container

`league_data` is a dictionary with one entry per league:

```python
league_data["pl"]
league_data["de"]
league_data["es"]
league_data["it"]
league_data["fr"]